<a href="https://colab.research.google.com/github/Praharshita1275/Agentic-Medical-Fact-Verification-system/blob/main/MEDICAL_CLAIM_VERIFIER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Medical Claim Verifier — Extended Version

| Feature | Description |
|---------|-------------|
| **E2** | Multi-Round Rebuttal Debate |
| **E3** | Live PubMed Search |
| **E4** | Uncertainty Quantification |

**Team:** G Akshatha · Gaali Sai Praharshita · Srichandana  
**Dept of IT, CBIT Hyderabad**


## Cell 1 — Install Dependencies

In [ ]:
!pip install -q \
    sentence-transformers \
    faiss-cpu \
    google-generativeai \
    requests numpy pandas tqdm \
    scikit-learn \
    biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.5 MB/s eta 0:00:00


## Cell 2 — Imports & Setup

In [ ]:
import os, re, json, time, pickle, requests
import numpy as np
import faiss
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import google.generativeai as genai
from Bio import Entrez                          # PubMed API (from biopython)

# ── Directories ──────────────────────────────────────────
for d in ["index", "data"]:
    os.makedirs(d, exist_ok=True)

# ── Gemini setup ─────────────────────────────────────────
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"          # ← paste your free key here
# Get free key at: aistudio.google.com → Get API Key
genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel("gemini-1.5-flash")

# ── PubMed / Entrez setup ────────────────────────────────
Entrez.email = "your-email@example.com"         # ← required by NCBI (any email)
# Free API key from: ncbi.nlm.nih.gov/account  (optional but increases rate limit)
NCBI_API_KEY  = ""                              # ← paste NCBI key here if you have one

# ── SentenceTransformer ──────────────────────────────────
MODEL_NAME    = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

print("Loading SentenceTransformer...")
st_model = SentenceTransformer(MODEL_NAME)
print("✅ Setup complete")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Loading SentenceTransformer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Setup complete


## Cell 3 — Load PubHealth Dataset

Download the dataset from [HuggingFace](https://huggingface.co/datasets/pubhealth)  
Or run these in Colab first:
```bash
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/train.tsv
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/test.tsv
```


In [ ]:
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/train.tsv

In [ ]:
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/test.tsv

In [ ]:
from google.colab import files
uploaded = files.upload()   # click and select train.tsv and test.tsv


Saving dev.tsv to dev.tsv
Saving test.tsv to test.tsv
Saving train.tsv to train.tsv


In [ ]:
# ════════════════════════════════════════════════════════
# CELL 3 — Load PubHealth Dataset (from uploaded TSV)
# ════════════════════════════════════════════════════════

def load_pubhealth(path):
    """Load and preprocess PubHealth dataset."""
    df = pd.read_csv(path, sep="\t", on_bad_lines="skip", engine="python")

    # Keep only needed columns (test.tsv has no explanation column)
    available = [c for c in ["claim", "label", "explanation"] if c in df.columns]
    df = df[available].dropna()

    if "explanation" not in df.columns:
        df["explanation"] = ""

    # Map labels to 3 categories
    label_map = {"true": "true", "false": "false",
                 "mixture": "uncertain", "unproven": "uncertain"}
    df["label"] = df["label"].str.lower().map(label_map)
    df = df.dropna(subset=["label"])

    df["text"] = df["claim"].str.strip() + " " + df["explanation"].str.strip()
    return df[["claim", "label", "explanation", "text"]]

print("Loading PubHealth dataset...")
train_df = load_pubhealth("train.tsv")
test_df  = load_pubhealth("dev.tsv")    # dev.tsv has explanation; test.tsv does not
print(f"  Train: {len(train_df)} | Test: {len(test_df)}")
print("✅ Dataset loaded")

Loading PubHealth dataset...
  Train: 9804 | Test: 1214
✅ Dataset loaded


In [ ]:
def load_pubhealth(path):
    """Load and preprocess PubHealth dataset."""
    try:
        df = pd.read_csv(path, sep="\t", on_bad_lines="skip")
    except Exception:
        df = pd.read_csv(path, sep="\t", error_bad_lines=False)

    # Keep only needed columns
    df = df[["claim", "label", "explanation"]].dropna()

    # Map labels to 3 categories
    label_map = {"true": "true", "false": "false",
                 "mixture": "uncertain", "unproven": "uncertain"}
    df["label"] = df["label"].str.lower().map(label_map)
    df = df.dropna(subset=["label"])

    # Combine claim + explanation as corpus text
    df["text"] = df["claim"].str.strip() + " " + df["explanation"].str.strip()
    return df

# ── Load datasets ──────────────────────────────────────
print("Loading PubHealth dataset...")
try:
    train_df = load_pubhealth("train.tsv")
    test_df  = load_pubhealth("test.tsv")
    print(f"  Train: {len(train_df)} | Test: {len(test_df)}")
except FileNotFoundError:
    print("  ⚠️  TSV files not found.")
    print("  Run these in Colab first:")
    print("  !wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/train.tsv")
    print("  !wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/test.tsv")
    # Fallback: small hardcoded sample so rest of code runs
    sample_data = [
        {"claim": "Vaccines cause autism.", "label": "false",
         "explanation": "Multiple large studies have found no link between vaccines and autism.", "text": ""},
        {"claim": "Drinking water helps prevent kidney stones.", "label": "true",
         "explanation": "Adequate hydration reduces risk of kidney stone formation.", "text": ""},
        {"claim": "COVID-19 vaccines contain microchips.", "label": "false",
         "explanation": "There is no evidence of microchips in COVID-19 vaccines.", "text": ""},
        {"claim": "Exercise reduces risk of heart disease.", "label": "true",
         "explanation": "Regular physical activity strengthens the heart and reduces cardiovascular risk.", "text": ""},
        {"claim": "Vitamin C cures the common cold.", "label": "uncertain",
         "explanation": "Some studies show modest benefit but evidence is mixed.", "text": ""},
    ]
    for r in sample_data:
        r["text"] = r["claim"] + " " + r["explanation"]
    train_df = pd.DataFrame(sample_data)
    test_df  = train_df.copy()
    print("  Using fallback sample data.")

print("✅ Dataset loaded")


Loading PubHealth dataset...
  Train: 9804 | Test: 1233
✅ Dataset loaded


## Cell 4 — Build FAISS Index from PubHealth

In [ ]:
corpus_records = train_df[["claim", "label", "text"]].to_dict("records")

print(f"Encoding {len(corpus_records)} PubHealth passages...")
corpus_texts = [r["text"] for r in corpus_records]
embeddings   = st_model.encode(corpus_texts, batch_size=64,
                                show_progress_bar=True,
                                convert_to_numpy=True).astype(np.float32)

index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings)
faiss.write_index(index, "index/pubhealth_index.bin")
with open("index/pubhealth_metadata.pkl", "wb") as f:
    pickle.dump(corpus_records, f)

print(f"✅ FAISS index built — {index.ntotal} vectors")


Encoding 9804 PubHealth passages...


Batches:   0%|          | 0/154 [00:00<?, ?it/s]

✅ FAISS index built — 9804 vectors


## Cell 5 — E3: Live PubMed Search

In [ ]:
def search_pubmed(query, max_results=5):
    """
    Search PubMed live for a query and return abstracts.
    Uses NCBI Entrez API — completely free.
    """
    results = []
    try:
        # Step 1: Search for article IDs
        search_params = {
            "db": "pubmed",
            "term": query,
            "retmax": max_results,
            "sort": "relevance",
        }
        if NCBI_API_KEY:
            search_params["api_key"] = NCBI_API_KEY

        search_handle = Entrez.esearch(**search_params)
        search_record = Entrez.read(search_handle)
        search_handle.close()
        id_list = search_record.get("IdList", [])

        if not id_list:
            return results

        # Step 2: Fetch abstracts for those IDs
        fetch_handle = Entrez.efetch(
            db="pubmed", id=",".join(id_list),
            rettype="abstract", retmode="xml"
        )
        fetch_record = Entrez.read(fetch_handle)
        fetch_handle.close()

        for article in fetch_record.get("PubmedArticle", []):
            try:
                medline  = article["MedlineCitation"]
                art_data = medline["Article"]
                title    = str(art_data.get("ArticleTitle", ""))
                abstract_obj = art_data.get("Abstract", {})
                abstract_texts = abstract_obj.get("AbstractText", [])
                abstract = " ".join([str(a) for a in abstract_texts])
                if abstract and len(abstract) > 50:
                    results.append({
                        "id":       medline.get("PMID", ""),
                        "title":    title,
                        "text":     f"{title}. {abstract[:500]}",
                        "source":   "PubMed",
                        "label":    "pubmed_article",
                        "severity": "INFO"
                    })
            except Exception:
                continue

        time.sleep(0.4)   # respect NCBI rate limits (3 req/sec without key, 10/sec with key)

    except Exception as e:
        print(f"  PubMed warning: {e}")

    return results


def retrieve_faiss(claim, top_k=5):
    """Retrieve top-k passages from local FAISS index."""
    q = st_model.encode([claim], convert_to_numpy=True,
                         show_progress_bar=False).astype(np.float32)
    dists, idxs = index.search(q, k=top_k)
    return [
        {**corpus_records[i], "rank": r + 1, "distance": float(d), "source": "PubHealth"}
        for r, (i, d) in enumerate(zip(idxs[0], dists[0])) if i != -1
    ]


def retrieve_all(claim, faiss_k=5, pubmed_k=3):
    """
    E3: Combined retrieval — local FAISS + live PubMed.
    Returns merged, deduplicated list of evidence passages.
    """
    faiss_results  = retrieve_faiss(claim, top_k=faiss_k)
    pubmed_results = search_pubmed(claim + " medical evidence", max_results=pubmed_k)
    combined = faiss_results + pubmed_results
    # Deduplicate by text prefix
    seen, unique = set(), []
    for r in combined:
        key = r["text"][:80]
        if key not in seen:
            seen.add(key)
            unique.append(r)
    return unique


print("✅ E3 — PubMed retrieval ready")


✅ E3 — PubMed retrieval ready


## Cell 6 — Evidence Classification (Gemini-powered)

In [ ]:
def classify_evidence_gemini(claim, passages):
    """
    Use Gemini to classify each passage as SUPPORT, OPPOSE, or NEUTRAL.
    Much more accurate than VADER for medical text.
    """
    pos_ev, neg_ev = [], []
    for p in passages:
        prompt = f"""You are a medical evidence classifier.

Claim: "{claim}"
Evidence: "{p['text'][:400]}"

Does this evidence SUPPORT, OPPOSE, or is it NEUTRAL toward the claim?
Reply with exactly ONE word: SUPPORT, OPPOSE, or NEUTRAL."""
        try:
            label = gemini.generate_content(prompt).text.strip().upper()
            if "SUPPORT" in label:
                pos_ev.append({**p, "gemini_label": "SUPPORT"})
            elif "OPPOSE" in label:
                neg_ev.append({**p, "gemini_label": "OPPOSE"})
        except Exception as e:
            print(f"  classify warning: {e}")
        time.sleep(0.3)

    return pos_ev, neg_ev


print("✅ Evidence classifier ready")


✅ Evidence classifier ready


## Cell 7 — E2: Multi-Round Rebuttal Debate Agents

In [ ]:
def pro_agent_round1(claim, pos_ev):
    """Pro Agent — Round 1: Build initial supporting argument."""
    if not pos_ev:
        return {"argument": "Insufficient supporting evidence found.", "score": 0.0}

    ev_text = "\n\n".join([
        f"[Evidence {i+1} | {e.get('source', '')}]\n{e['text'][:300]}"
        for i, e in enumerate(pos_ev[:4])
    ])
    prompt = f"""You are a medical debate agent arguing IN FAVOUR of the claim.
Use ONLY the evidence below. No outside knowledge.

Claim: "{claim}"
Evidence:
{ev_text}

Build the strongest supporting argument. Cite evidence by number.
End with a clear conclusion. Max 200 words.
Supporting argument:"""
    try:
        arg   = gemini.generate_content(prompt).text.strip()
        score = len(pos_ev) * 0.35 + min(len(arg) / 400, 1.0)
        return {"argument": arg, "score": score}
    except Exception as e:
        return {"argument": f"Error: {e}", "score": 0.0}


def con_agent_round1(claim, neg_ev):
    """Con Agent — Round 1: Build initial opposing argument."""
    if not neg_ev:
        return {"argument": "Insufficient opposing evidence found.", "score": 0.0}

    ev_text = "\n\n".join([
        f"[Evidence {i+1} | {e.get('source', '')}]\n{e['text'][:300]}"
        for i, e in enumerate(neg_ev[:4])
    ])
    prompt = f"""You are a medical debate agent arguing AGAINST the claim.
Use ONLY the evidence below. No outside knowledge.

Claim: "{claim}"
Evidence:
{ev_text}

Build the strongest opposing argument. Cite evidence by number.
End with a clear conclusion. Max 200 words.
Opposing argument:"""
    try:
        arg   = gemini.generate_content(prompt).text.strip()
        score = len(neg_ev) * 0.35 + min(len(arg) / 400, 1.0)
        return {"argument": arg, "score": score}
    except Exception as e:
        return {"argument": f"Error: {e}", "score": 0.0}


def pro_rebuttal(claim, pro_arg, con_arg):
    """E2 — Pro Agent Round 2: Read Con's argument and rebut it."""
    prompt = f"""You are a medical debate agent arguing IN FAVOUR of this claim.

Claim: "{claim}"

Your previous argument:
{pro_arg['argument'][:400]}

Your opponent's counter-argument:
{con_arg['argument'][:400]}

Write a rebuttal that directly addresses the opponent's points and
strengthens your position. Max 150 words.
Your rebuttal:"""
    try:
        arg   = gemini.generate_content(prompt).text.strip()
        score = pro_arg["score"] + 0.2
        return {"argument": arg, "score": score}
    except Exception as e:
        return {"argument": f"Error: {e}", "score": pro_arg["score"]}


def con_rebuttal(claim, pro_arg, con_arg):
    """E2 — Con Agent Round 2: Read Pro's argument and rebut it."""
    prompt = f"""You are a medical debate agent arguing AGAINST this claim.

Claim: "{claim}"

Your previous argument:
{con_arg['argument'][:400]}

Your opponent's counter-argument:
{pro_arg['argument'][:400]}

Write a rebuttal that directly addresses the opponent's points and
strengthens your opposition. Max 150 words.
Your rebuttal:"""
    try:
        arg   = gemini.generate_content(prompt).text.strip()
        score = con_arg["score"] + 0.2
        return {"argument": arg, "score": score}
    except Exception as e:
        return {"argument": f"Error: {e}", "score": con_arg["score"]}


print("✅ E2 — Multi-round debate agents ready")


✅ E2 — Multi-round debate agents ready


## Cell 8 — E4: Uncertainty Quantification

In [ ]:
def quantify_uncertainty(result, pos_ev, neg_ev, all_passages):
    """
    E4 — Uncertainty Quantification.
    Diagnoses WHY the system is uncertain and scores each reason.

    Uncertainty sources:
      - Evidence conflict   : both sides have strong evidence
      - Evidence scarcity   : very few passages retrieved
      - Claim ambiguity     : Gemini flags the claim as vague/complex
      - Score proximity     : pro and con scores are very close
      - Source diversity    : only one source type (only PubMed or only PubHealth)
    """
    uncertainty_report = {}
    total_uncertainty  = 0.0

    # 1. Evidence conflict — both sides have evidence
    conflict_score = 0.0
    if len(pos_ev) > 0 and len(neg_ev) > 0:
        ratio = min(len(pos_ev), len(neg_ev)) / max(len(pos_ev), len(neg_ev))
        conflict_score = round(ratio * 0.4, 3)   # max 0.4
    uncertainty_report["evidence_conflict"] = {
        "score":  conflict_score,
        "detail": f"{len(pos_ev)} supporting vs {len(neg_ev)} opposing passages"
    }
    total_uncertainty += conflict_score

    # 2. Evidence scarcity — too few passages
    scarcity_score = 0.0
    if len(all_passages) == 0:
        scarcity_score = 0.4
    elif len(all_passages) < 3:
        scarcity_score = round((3 - len(all_passages)) / 3 * 0.3, 3)
    uncertainty_report["evidence_scarcity"] = {
        "score":  scarcity_score,
        "detail": f"Only {len(all_passages)} passages retrieved"
    }
    total_uncertainty += scarcity_score

    # 3. Score proximity — pro and con scores are too close
    proximity_score = 0.0
    ps = result.get("pro_score", 0)
    cs = result.get("con_score", 0)
    total = ps + cs + 0.001
    diff  = abs(ps - cs) / total
    if diff < 0.15:
        proximity_score = round((0.15 - diff) / 0.15 * 0.2, 3)   # max 0.2
    uncertainty_report["score_proximity"] = {
        "score":  proximity_score,
        "detail": f"Pro={ps:.2f} vs Con={cs:.2f} (diff ratio={diff:.2f})"
    }
    total_uncertainty += proximity_score

    # 4. Source diversity — penalise if all evidence from same source
    sources = [p.get("source", "") for p in all_passages]
    unique_sources = len(set(sources))
    diversity_score = 0.0
    if unique_sources == 1 and len(all_passages) > 0:
        diversity_score = 0.1
    uncertainty_report["source_diversity"] = {
        "score":  diversity_score,
        "detail": f"Sources used: {list(set(sources))}"
    }
    total_uncertainty += diversity_score

    # 5. Claim ambiguity — ask Gemini if the claim itself is ambiguous
    ambiguity_score = 0.0
    try:
        prompt = f"""Is this medical claim ambiguous, multi-part, or hard to verify clearly?
Claim: "{result.get('claim', '')}"
Reply with ONE word: YES or NO."""
        resp = gemini.generate_content(prompt).text.strip().upper()
        if "YES" in resp:
            ambiguity_score = 0.15
        uncertainty_report["claim_ambiguity"] = {
            "score":  ambiguity_score,
            "detail": "Gemini flagged claim as ambiguous" if ambiguity_score > 0 else "Claim appears clear"
        }
        total_uncertainty += ambiguity_score
        time.sleep(0.3)
    except Exception:
        uncertainty_report["claim_ambiguity"] = {"score": 0.0, "detail": "Could not assess"}

    # Normalise total to 0–100
    total_uncertainty = min(total_uncertainty, 1.0)
    uncertainty_pct   = round(total_uncertainty * 100, 1)

    # Uncertainty level label
    if uncertainty_pct < 20:
        level = "LOW"
    elif uncertainty_pct < 50:
        level = "MEDIUM"
    else:
        level = "HIGH"

    return {
        "uncertainty_score":   uncertainty_pct,
        "uncertainty_level":   level,
        "uncertainty_sources": uncertainty_report,
        "interpretation": (
            f"Uncertainty is {level} ({uncertainty_pct}%). "
            + ("The verdict is reliable." if level == "LOW"
               else "Treat verdict with caution." if level == "MEDIUM"
               else "Verdict is unreliable — insufficient or conflicting evidence.")
        )
    }


print("✅ E4 — Uncertainty quantification ready")


✅ E4 — Uncertainty quantification ready


## Cell 9 — Judge Agent (reads all 4 arguments)

In [ ]:
def judge_agent(claim, pro_r1, con_r1, pro_r2, con_r2):
    """
    Judge reads all 4 arguments (2 rounds each side) and gives final verdict.
    """
    prompt = f"""You are an impartial medical fact-checking judge.
You have read a full 2-round debate between two agents.

Claim: "{claim}"

=== ROUND 1 ===
PRO (supporting): {pro_r1.get('argument', '')[:300]}
CON (opposing):   {con_r1.get('argument', '')[:300]}

=== ROUND 2 (Rebuttals) ===
PRO rebuttal: {pro_r2.get('argument', '')[:300]}
CON rebuttal: {con_r2.get('argument', '')[:300]}

Evaluate all four arguments carefully. Respond EXACTLY in this format:
VERDICT: [TRUE/FALSE/UNCERTAIN]
CONFIDENCE: [0-100]
REASONING: [3-4 sentence chain-of-thought explanation]"""

    try:
        text = gemini.generate_content(prompt).text.strip()
        verdict, confidence, reasoning = "Uncertain", 50.0, ""
        for line in text.split("\n"):
            if line.startswith("VERDICT:"):
                v = line.replace("VERDICT:", "").strip().upper()
                verdict = "True" if "TRUE" in v else ("False" if "FALSE" in v else "Uncertain")
            elif line.startswith("CONFIDENCE:"):
                try:
                    confidence = float(re.findall(r"[\d.]+", line)[0])
                except Exception:
                    pass
            elif line.startswith("REASONING:"):
                reasoning = line.replace("REASONING:", "").strip()
        ps = (pro_r1["score"] + pro_r2["score"]) / 2
        cs = (con_r1["score"] + con_r2["score"]) / 2
        return {
            "claim":           claim,
            "verdict":         verdict,
            "confidence":      confidence,
            "reason":          reasoning,
            "pro_score":       round(ps, 3),
            "con_score":       round(cs, 3),
            "pro_r1":          pro_r1["argument"],
            "con_r1":          con_r1["argument"],
            "pro_rebuttal":    pro_r2["argument"],
            "con_rebuttal":    con_r2["argument"],
        }
    except Exception as e:
        ps = (pro_r1["score"] + pro_r2["score"]) / 2
        cs = (con_r1["score"] + con_r2["score"]) / 2
        total = ps + cs + 0.001
        v = "True" if ps > cs else ("False" if cs > ps else "Uncertain")
        c = round(max(ps, cs) / total * 100, 2)
        return {
            "claim": claim, "verdict": v, "confidence": c,
            "reason": f"Fallback scoring (Gemini error: {e})",
            "pro_score": round(ps, 3), "con_score": round(cs, 3),
            "pro_r1": pro_r1["argument"], "con_r1": con_r1["argument"],
            "pro_rebuttal": pro_r2["argument"], "con_rebuttal": con_r2["argument"],
        }


print("✅ Judge agent ready")


✅ Judge agent ready


## Cell 10 — Full Pipeline: `verify_claim()`

In [ ]:
def verify_claim(claim, faiss_k=5, pubmed_k=3, verbose=True):
    """
    Full pipeline:
      1. Retrieve evidence (FAISS + live PubMed)          [E3]
      2. Classify evidence with Gemini
      3. Round 1 debate (Pro + Con)                       [E2]
      4. Round 2 rebuttals (Pro + Con)                    [E2]
      5. Judge reads all 4 arguments
      6. Uncertainty quantification                        [E4]
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"CLAIM: {claim}")
        print(f"{'='*60}")

    # Step 1 — Retrieve
    if verbose: print("Step 1: Retrieving evidence (FAISS + PubMed)...")
    all_passages = retrieve_all(claim, faiss_k=faiss_k, pubmed_k=pubmed_k)
    if verbose: print(f"  → {len(all_passages)} passages retrieved "
                      f"({sum(1 for p in all_passages if p.get('source')=='PubMed')} from PubMed live)")

    # Step 2 — Classify
    if verbose: print("Step 2: Classifying evidence with Gemini...")
    pos_ev, neg_ev = classify_evidence_gemini(claim, all_passages)
    if verbose: print(f"  → {len(pos_ev)} supporting | {len(neg_ev)} opposing")

    # Step 3 — Round 1
    if verbose: print("Step 3: Round 1 debate...")
    pro_r1 = pro_agent_round1(claim, pos_ev)
    con_r1 = con_agent_round1(claim, neg_ev)

    # Step 4 — Round 2 Rebuttals
    if verbose: print("Step 4: Round 2 rebuttals...")
    pro_r2 = pro_rebuttal(claim, pro_r1, con_r1)
    con_r2 = con_rebuttal(claim, pro_r1, con_r1)

    # Step 5 — Judge
    if verbose: print("Step 5: Judge evaluating all arguments...")
    result = judge_agent(claim, pro_r1, con_r1, pro_r2, con_r2)
    result["retrieved_count"]    = len(all_passages)
    result["pro_evidence_count"] = len(pos_ev)
    result["con_evidence_count"] = len(neg_ev)

    # Step 6 — Uncertainty
    if verbose: print("Step 6: Quantifying uncertainty...")
    uncertainty = quantify_uncertainty(result, pos_ev, neg_ev, all_passages)
    result.update(uncertainty)

    return result


def print_result(r):
    """Pretty-print a verification result."""
    icon   = {"True": "✅", "False": "❌", "Uncertain": "⚠️"}.get(r["verdict"], "⚠️")
    u_icon = {"LOW": "🟢", "MEDIUM": "🟡", "HIGH": "🔴"}.get(r["uncertainty_level"], "⚪")

    print(f"\n{'─'*60}")
    print(f"CLAIM      : {r['claim']}")
    print(f"VERDICT    : {icon} {r['verdict']}  (Confidence: {r['confidence']}%)")
    print(f"UNCERTAINTY: {u_icon} {r['uncertainty_level']} ({r['uncertainty_score']}%)")
    print(f"REASONING  : {r['reason'][:250]}")
    print(f"INTERPRET  : {r['interpretation']}")
    print(f"\n── Evidence ──")
    print(f"  Retrieved : {r['retrieved_count']} passages")
    print(f"  Supporting: {r['pro_evidence_count']}")
    print(f"  Opposing  : {r['con_evidence_count']}")
    print(f"\n── Uncertainty Breakdown ──")
    for k, v in r["uncertainty_sources"].items():
        bar = "█" * int(v["score"] * 20)
        print(f"  {k:<22}: {v['score']:.2f}  {bar}  {v['detail']}")
    print(f"\n── Debate Summary ──")
    print(f"  [PRO R1]  : {r['pro_r1'][:120]}...")
    print(f"  [CON R1]  : {r['con_r1'][:120]}...")
    print(f"  [PRO R2]  : {r['pro_rebuttal'][:120]}...")
    print(f"  [CON R2]  : {r['con_rebuttal'][:120]}...")
    print(f"{'─'*60}")


print("✅ Full pipeline ready")


✅ Full pipeline ready


## Cell 11 — Test on Sample Claims

In [ ]:
SAMPLE_CLAIMS = [
    "Vaccines cause autism.",
    "Exercise reduces the risk of heart disease.",
    "Vitamin C cures the common cold.",
    "COVID-19 vaccines are safe and effective.",
]

print("\n🔬 Running verification on sample claims...\n")
for claim in SAMPLE_CLAIMS:
    result = verify_claim(claim, faiss_k=5, pubmed_k=3, verbose=True)
    print_result(result)
    time.sleep(1)   # avoid hitting Gemini rate limit



🔬 Running verification on sample claims...


CLAIM: Vaccines cause autism.
Step 1: Retrieving evidence (FAISS + PubMed)...
  → 8 passages retrieved (3 from PubMed live)
Step 2: Classifying evidence with Gemini...


  classify warning: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


  classify warning: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


  classify warning: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


  classify warning: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


...

  classify warning: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.


  classify warning: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.
  → 0 supporting | 0 opposing
Step 3: Round 1 debate...
Step 4: Round 2 rebuttals...


Step 5: Judge evaluating all arguments...


Step 6: Quantifying uncertainty...



────────────────────────────────────────────────────────────
CLAIM      : COVID-19 vaccines are safe and effective.
VERDICT    : ⚠️ Uncertain  (Confidence: 0.0%)
UNCERTAINTY: 🟡 MEDIUM (20.0%)
REASONING  : Fallback scoring (Gemini error: 400 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: API key not valid. Please pass a valid API key.)
INTERPRET  : Uncertainty is MEDIUM (20.0%). Treat verdict with caution.

── Evidence ──
  Retrieved : 8 passages
  Supporting: 0
  Opposing  : 0

── Uncertainty Breakdown ──
  evidence_conflict     : 0.00    0 supporting vs 0 opposing passages
  evidence_scarcity     : 0.00    Only 8 passages retrieved
  score_proximity       : 0.20  ████  Pro=0.00 vs Con=0.00 (diff ratio=0.00)
  source_diversity      : 0.00    Sources used: ['PubHealth', 'PubMed']
  claim_ambiguity       : 0.00    Could not assess

── Debate Summary ──
  [PRO R1]  : Insufficient supporting evidence found..

## Cell 12 — Evaluation on PubHealth Test Set

In [ ]:
# Use a small subset for evaluation (full test set would use many API calls)
EVAL_SAMPLE_SIZE = 20

print(f"\nEvaluating on {EVAL_SAMPLE_SIZE} test claims...")
test_sample = test_df.sample(min(EVAL_SAMPLE_SIZE, len(test_df)),
                              random_state=42).reset_index(drop=True)

true_labels, pred_labels, confidences, uncertainty_scores = [], [], [], []

for _, row in tqdm(test_sample.iterrows(), total=len(test_sample), desc="Evaluating"):
    try:
        r = verify_claim(row["claim"], faiss_k=5, pubmed_k=2, verbose=False)
        pred_labels.append(r["verdict"].lower())
        confidences.append(r["confidence"])
        uncertainty_scores.append(r["uncertainty_score"])
        true_labels.append(row["label"])
    except Exception as e:
        print(f"  Error on claim: {e}")
        pred_labels.append("uncertain")
        confidences.append(50.0)
        uncertainty_scores.append(50.0)
        true_labels.append(row["label"])
    time.sleep(0.5)

LABEL_ORDER = ["true", "false", "uncertain"]

acc = accuracy_score(true_labels, pred_labels) * 100
f1  = f1_score(true_labels, pred_labels, average="macro",
               labels=LABEL_ORDER, zero_division=0) * 100

print(f"\n{'System':<40} {'Acc%':>6} {'F1%':>6}")
print("─" * 55)
print(f"{'Extended (Gemini + PubMed + 2-Round + UQ)':<40} {acc:>6.1f} {f1:>6.1f}")
print(f"\nAverage Confidence Score : {np.mean(confidences):.1f}%")
print(f"Average Uncertainty Score: {np.mean(uncertainty_scores):.1f}%")

print(f"\nDetailed Classification Report:")
print(classification_report(true_labels, pred_labels,
                             labels=LABEL_ORDER, zero_division=0))


## Cell 13 — Interactive: Verify Any Claim

Change `YOUR_CLAIM` below and run the cell.


In [ ]:
YOUR_CLAIM = "Drinking coffee reduces the risk of Alzheimer's disease."

result = verify_claim(YOUR_CLAIM, faiss_k=5, pubmed_k=3, verbose=True)
print_result(result)


## Cell 14 — Save Results to JSON

In [ ]:
output_path = "data/verification_results.json"
results_to_save = []

for claim in SAMPLE_CLAIMS:
    try:
        r = verify_claim(claim, faiss_k=5, pubmed_k=2, verbose=False)
        # Remove long debate text for cleaner JSON
        save_r = {k: v for k, v in r.items()
                  if k not in ["pro_r1", "con_r1", "pro_rebuttal", "con_rebuttal"]}
        results_to_save.append(save_r)
        time.sleep(0.5)
    except Exception as e:
        print(f"  Save error: {e}")

with open(output_path, "w") as f:
    json.dump(results_to_save, f, indent=2)

print(f"✅ Results saved → {output_path}")
print("\n✅ ALL CELLS COMPLETE")
print("  E2 (Multi-round rebuttal debate)  — ✅")
print("  E3 (Live PubMed search)            — ✅")
print("  E4 (Uncertainty quantification)    — ✅")
